# Section 4 — Results & Evaluation Figures

This notebook generates all six figures used in Section 4 of the report.  
Output is saved to the same `results_evaluation/` folder.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.patches import Patch
import os

OUT = os.path.dirname(os.path.abspath('__file__'))  # results_evaluation/

# ── ICLR-style palette ──
BLUE  = '#4878A8'
RED   = '#C05050'
GREEN = '#5A9E6F'
GREY  = '#7A7A7A'
LGREY = '#B0B0B0'

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.facecolor': '#FFFFFF',
    'figure.facecolor': '#FFFFFF',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': '#CCCCCC',
    'grid.linewidth': 0.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'legend.frameon': True,
    'legend.edgecolor': '#CCCCCC',
    'legend.fancybox': False,
})

print('Setup complete')

## Figure 1 — Test-Set Macro F1 by Model

In [ ]:
models = ['Few-shot\nHaiku 4.5', 'Frozen SBERT\n+ LR', 'TF-IDF\n+ LR', 'Fine-tuned\nDeBERTa']
urg_f1 = [0.535, 0.691, 0.819, 0.803]
emo_f1 = [0.497, 0.631, 0.764, 0.857]

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(models))
w = 0.30
b1 = ax.bar(x - w/2, urg_f1, w, label='Urgency Macro F1', color=BLUE, edgecolor='#333333', linewidth=0.5)
b2 = ax.bar(x + w/2, emo_f1, w, label='Emotion Macro F1', color=RED, edgecolor='#333333', linewidth=0.5)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Macro F1 Score')
ax.set_title('Test-Set Macro F1 by Model', fontsize=11, pad=10)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylim(0, 1.0)
ax.axhline(y=0.333, color=LGREY, linestyle=':', linewidth=1)
ax.text(3.42, 0.343, 'Random', fontsize=7.5, color=GREY)

fig.subplots_adjust(bottom=0.25)
fig.legend(*ax.get_legend_handles_labels(), loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 0.02), frameon=True, edgecolor='#CCCCCC')

fig.savefig(f'{OUT}/fig1_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2 — Per-Class F1 Scores

In [ ]:
plt.rcParams.update({'font.size': 13})  # larger base font for this figure

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
classes = ['Low', 'Medium', 'High']
colours = [LGREY, GREEN, BLUE, RED]

urg_data = {'Haiku 4.5': [0.416, 0.529, 0.661], 'SBERT+LR': [0.696, 0.628, 0.750],
            'TF-IDF+LR': [0.803, 0.781, 0.874], 'DeBERTa': [0.835, 0.734, 0.839]}
emo_data = {'Haiku 4.5': [0.552, 0.381, 0.558], 'SBERT+LR': [0.696, 0.567, 0.630],
            'TF-IDF+LR': [0.819, 0.711, 0.761], 'DeBERTa': [0.900, 0.813, 0.857]}

all_handles, all_labels = None, None
for ax, data, title in [(axes[0], urg_data, '(a) Urgency Per-Class F1'),
                         (axes[1], emo_data, '(b) Emotion Per-Class F1')]:
    x = np.arange(len(classes))
    w = 0.18
    for i, (name, vals) in enumerate(data.items()):
        offset = (i - 1.5) * w
        bars = ax.bar(x + offset, vals, w, label=name, color=colours[i], edgecolor='#333333', linewidth=0.4)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=13)
    ax.tick_params(axis='y', labelsize=12)
    ax.set_title(title, fontsize=14, pad=8)
    ax.set_ylim(0, 1.05)
    if all_handles is None:
        all_handles, all_labels = ax.get_legend_handles_labels()

axes[0].set_ylabel('F1 Score', fontsize=13)
fig.suptitle('Per-Class F1 Scores on Test Set', fontsize=15)

fig.subplots_adjust(bottom=0.15)
fig.legend(all_handles, all_labels, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, 0.01), frameon=True, edgecolor='#CCCCCC')

fig.savefig(f'{OUT}/fig2_per_class_f1.png', dpi=300, bbox_inches='tight')
plt.show()

plt.rcParams.update({'font.size': 10})  # reset

## Figure 3 — DeBERTa Training Dynamics

In [ ]:
epochs = list(range(1, 10))
train_loss = [2.111, 1.330, 0.915, 0.612, 0.411, 0.271, 0.155, 0.097, 0.053]
val_loss   = [1.642, 1.147, 0.957, 1.083, 1.145, 1.241, 1.500, 1.630, 1.745]
val_urg_f1 = [0.616, 0.751, 0.792, 0.788, 0.808, 0.826, 0.818, 0.805, 0.803]
val_emo_f1 = [0.551, 0.796, 0.834, 0.803, 0.835, 0.834, 0.832, 0.834, 0.834]
val_comb   = [0.584, 0.773, 0.813, 0.795, 0.821, 0.830, 0.825, 0.819, 0.818]

fig, ax1 = plt.subplots(figsize=(7.5, 5))

ax1.plot(epochs, train_loss, 's--', color=LGREY, markersize=4, linewidth=1.2, label='Train Loss')
ax1.plot(epochs, val_loss, 'o--', color=GREY, markersize=4, linewidth=1.2, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='#555555')
ax1.tick_params(axis='y', labelcolor='#555555')
ax1.set_ylim(0, 2.2)

ax2 = ax1.twinx()
ax2.plot(epochs, val_urg_f1, 'o-', color=BLUE, markersize=5, linewidth=1.5, label='Val Urgency F1')
ax2.plot(epochs, val_emo_f1, 's-', color=RED, markersize=5, linewidth=1.5, label='Val Emotion F1')
ax2.plot(epochs, val_comb, '^-', color='#333333', markersize=5, linewidth=1.5, label='Val Combined F1')
ax2.set_ylabel('Macro F1')
ax2.set_ylim(0.4, 0.90)

ax2.axvline(x=6, color=LGREY, linestyle=':', linewidth=1.2)
ax2.annotate('Best epoch (6)', xy=(6, 0.830), xytext=(7.3, 0.865),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='#666666', lw=0.8), color='#333333')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.set_title('DeBERTa Training Dynamics', fontsize=11, pad=10)
ax1.set_xticks(epochs)

fig.subplots_adjust(bottom=0.18)
fig.legend(lines1 + lines2, labels1 + labels2, loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, 0.01), frameon=True, edgecolor='#CCCCCC')

fig.savefig(f'{OUT}/fig3_training_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4 — Dataset Versioning Comparison

In [ ]:
versions = ['V1\n(vocab-constrained)', 'V2\n(tone-based, 4o-mini)', 'V3\n(GPT-5-mini, batch=5)']
v_urg = [0.578, 0.622, 0.803]
v_emo = [0.991, 0.835, 0.857]

fig, ax = plt.subplots(figsize=(6.5, 5))
x = np.arange(len(versions))
w = 0.28
b1 = ax.bar(x - w/2, v_urg, w, label='Urgency Macro F1', color=BLUE, edgecolor='#333333', linewidth=0.5)
b2 = ax.bar(x + w/2, v_emo, w, label='Emotion Macro F1', color=RED, edgecolor='#333333', linewidth=0.5)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8.5)

ax.annotate('Lexical leakage', xy=(0.14, 0.991), xytext=(0.75, 0.96),
            fontsize=8, fontstyle='italic', color='#555555',
            arrowprops=dict(arrowstyle='->', color='#888888', lw=0.8))

ax.set_ylabel('Macro F1 Score')
ax.set_title('DeBERTa Performance Across Dataset Versions', fontsize=11, pad=10)
ax.set_xticks(x)
ax.set_xticklabels(versions, fontsize=9)
ax.set_ylim(0, 1.1)

fig.subplots_adjust(bottom=0.22)
fig.legend(*ax.get_legend_handles_labels(), loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 0.02), frameon=True, edgecolor='#CCCCCC')

fig.savefig(f'{OUT}/fig4_dataset_versioning.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 5 — DeBERTa Confusion Matrices

In [ ]:
urg_cm = np.array([[235, 25, 2], [64, 203, 33], [2, 25, 161]])
emo_cm = np.array([[229, 15, 6], [19, 195, 36], [11, 20, 219]])
labels = ['Low', 'Medium', 'High']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, cm, title in [(axes[0], urg_cm, '(a) Urgency'), (axes[1], emo_cm, '(b) Emotion')]:
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cmap = mcolors.LinearSegmentedColormap.from_list('iclr', ['#F5F5F5', '#D6E4F0', BLUE])

    ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1, aspect='equal')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=10)

    for i in range(3):
        for j in range(3):
            colour = 'white' if cm_norm[i, j] > 0.6 else '#333333'
            ax.text(j, i, f'{cm[i,j]}', ha='center', va='center',
                   fontsize=12, fontweight='bold', color=colour)

fig.suptitle('DeBERTa Confusion Matrices', fontsize=11, y=1.02)
plt.tight_layout()
fig.savefig(f'{OUT}/fig5_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 6 — Adversarial Probe Results

In [ ]:
cases = ['1. Sarcastic\nPraise', '2. Polite\nCatastrophe', '3. Aggressive\nTriviality',
         '4. Legal\nThreat', '5. Rambling\nConfusion', '6. Urgent\nCheerful',
         '7. Sincere\nHeartbreak', '8. Machine\nLogic', '9. Ambiguous\nBroken',
         '10. Overreaction\nto Fix']
urg_correct = [0, 1, 1, 0, 1, 1, 1, 1, 0, 0]
emo_correct = [0, 1, 1, 0, 1, 1, 1, 1, 0, 0]

fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(cases))
w = 0.32

colours_urg = [BLUE if c else '#E0E0E0' for c in urg_correct]
colours_emo = [RED if c else '#E0E0E0' for c in emo_correct]

ax.bar(x - w/2, [1]*10, w, color=colours_urg, edgecolor='#999999', linewidth=0.5)
ax.bar(x + w/2, [1]*10, w, color=colours_emo, edgecolor='#999999', linewidth=0.5)

for i in range(10):
    sym_u = 'Y' if urg_correct[i] else 'N'
    sym_e = 'Y' if emo_correct[i] else 'N'
    col_u = 'white' if urg_correct[i] else '#999999'
    col_e = 'white' if emo_correct[i] else '#999999'
    ax.text(x[i] - w/2, 0.5, f'U:{sym_u}', ha='center', va='center',
           fontsize=9, fontweight='bold', color=col_u)
    ax.text(x[i] + w/2, 0.5, f'E:{sym_e}', ha='center', va='center',
           fontsize=9, fontweight='bold', color=col_e)

ax.set_xticks(x)
ax.set_xticklabels(cases, fontsize=7.5)
ax.set_yticks([])
ax.set_title('Adversarial Probe Results (U = Urgency, E = Emotion)', fontsize=11, pad=10)

fig.subplots_adjust(bottom=0.28)
ax.text(0.5, -0.18, '6 / 10 passed (both dimensions correct)', transform=ax.transAxes,
       ha='center', fontsize=9.5, color='#333333')

legend_elements = [Patch(facecolor=BLUE, edgecolor='#999999', label='Urgency correct'),
                   Patch(facecolor=RED, edgecolor='#999999', label='Emotion correct'),
                   Patch(facecolor='#E0E0E0', edgecolor='#999999', label='Incorrect')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=8,
           bbox_to_anchor=(0.5, 0.01), frameon=True, edgecolor='#CCCCCC')

fig.savefig(f'{OUT}/fig6_adversarial_probes.png', dpi=300, bbox_inches='tight')
plt.show()